# AI-Based Wildfire Occurrence Risk Assessment
## Bayesian Model Averaging · Application project · Unit 5

**Research question:** Can environmental conditions predict whether a recorded fire has a strictly positive recorded burned area?

**Deliverable:** a reproducible comparative experiment plus a functional Streamlit probability interface. This notebook runs the same source code used by the app and regenerates measured tables, plots and saved models.

### Target definition comes first
UCI Forest Fires was created for burned-area regression, with 517 fire records from Montesinho Natural Park, Portugal. It does not supply representative non-fire observation days. We define `y = 1[area > 0]` **before examining model performance**. A zero recorded area must not be interpreted as a confirmed non-fire day. The estimand is a **positive recorded-area proxy conditional on a fire record**, not population ignition risk. Actual occurrence prediction requires event/non-event sampling with exposure windows.

**Attribution:** Cortez, P. & Morais, A. (2007). *Forest Fires*. UCI Machine Learning Repository. [DOI: 10.24432/C5D88D](https://doi.org/10.24432/C5D88D). Dataset license: CC BY 4.0. [Official data page](https://archive.ics.uci.edu/dataset/162/forest%2Bfires).

### Reproducibility
From the project folder: `python -m pip install -r requirements.txt -c requirements-lock.txt`, then open this notebook. Run all cells in order. Training is deterministic given the package environment; timing values vary by hardware. Run `python -m wildfire.deliverables` after training to regenerate the short report and editable presentation. Python 3.13 is the tested runtime.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/wildfire').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from wildfire.data import load_data, feature_frame, grouped_split, RAW_FEATURES, FEATURES
from wildfire.train import run_experiment
df = load_data()
y = (df['area'] > 0).astype(int)
print('Project:', ROOT)
print('Rows / columns:', df.shape)
display(df.head())

## 1. Audit the source data
Keep the original records and original CSV. Exact duplicates are described rather than removed: all identical `(X,Y)` locations are assigned together, so identical records cannot cross splits. Rows may still share weather events across cells, an unresolved dependence limitation. Missing/invalid values fail validation rather than being silently filled from the full dataset.

In [ ]:
audit = pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'unique': df.nunique()})
display(audit)
print('Exact duplicate rows:', df.duplicated().sum())
display(pd.DataFrame({'target': ['area = 0', 'area > 0'], 'count': [(y == 0).sum(), (y == 1).sum()]}))
display(df[RAW_FEATURES[:-1] + ['area']].describe().T)
assert df.shape == (517, 13)
assert set(y.unique()) == {0, 1}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].hist(df.temp, bins=20, color='#377c75'); axes[0].set(xlabel='Temperature (°C)', ylabel='Records')
axes[1].hist(df.RH, bins=20, color='#377c75'); axes[1].set(xlabel='Relative humidity (%)')
axes[2].hist(np.log1p(df.area), bins=20, color='#c85b32'); axes[2].set(xlabel='log(1 + recorded area)')
fig.suptitle('Descriptive full-data audit; no modeling choices selected from test outcomes')
plt.tight_layout()
from io import BytesIO
buffer = BytesIO(); fig.savefig(buffer, format='png', dpi=120); plt.close(fig)
display(Image(data=buffer.getvalue()))

## 2. Prespecified spatial holdouts
Use `(X,Y)` as the repeated-location group. `GroupShuffleSplit(seed=42)` holds out 20% of cells for test; seed 43 holds out 25% of remaining cells for validation. These are group proportions, not row proportions. Do not try many seeds and keep the best result.

Month/day names do not provide full timestamps or years. Sorting alphabetic month/day names would invent a chronology, so this project makes no temporal generalization claim. X/Y are excluded from model inputs, because this experiment measures transfer to unseen grid cells.

Four-fold grouped CV uses **training only** for hyperparameter selection. Validation later selects classification thresholds. Test outcomes never determine preprocessing, hyperparameters, thresholds, model evidence or model weights.

**Revision disclosure:** the original repository already published this fixed test set. The expanded grids are selected only inside training, but this is not a fresh external validation. No test-driven seed search or target redefinition is performed.

In [ ]:
splits, groups = grouped_split(df)
summary = pd.DataFrame([{'split': name, 'records': len(ids), 'groups': groups.iloc[ids].nunique(),
                         'positive_fraction': y.iloc[ids].mean()} for name, ids in splits.items()])
display(summary)
group_sets = [set(groups.iloc[ids]) for ids in splits.values()]
assert all(group_sets[i].isdisjoint(group_sets[j]) for i in range(3) for j in range(i+1, 3))
assert len(set(np.concatenate(list(splits.values())))) == len(df)

## 3. Environmental preprocessing
Inputs: temperature (°C), RH (%), wind (km/h), rain (UCI `mm/m²` units), FFMC, DMC, DC, ISI and month. Rain is transformed by `log1p` and month has deterministic cyclic sine/cosine encoding. These transformations do not learn from data. Standardization learns mean/scale **inside each training fold** through a scikit-learn Pipeline, and is refit on training only for the selected model. Trees receive the same feature matrix for comparison.

Burned area enters only the label. X/Y enter only grouping; day of week is excluded. FWI components are provided measurements; the application does not claim to compute them from one day's weather.

In [ ]:
X = feature_frame(df)
display(X.head())
assert list(X.columns) == FEATURES
assert not {'area', 'X', 'Y', 'day'} & set(X.columns)
assert np.isfinite(X.to_numpy()).all()

## 4. Bayesian Model Averaging
The candidate family contains every subset of four prespecified feature blocks, for $2^4=16$ logistic models, including an intercept-only model:

| Block | Inputs |
|---|---|
| weather | temperature, RH, wind, log(1 + rain) |
| moisture | DMC, DC |
| fire_indices | FFMC, ISI |
| season | month sine, month cosine |

For model $M_m$, $y_i\sim\mathrm{Bernoulli}(\sigma(x_{im}^T\beta_m))$ and $\beta_m\sim N(0,s^2I)$, including the intercept. Inputs are standardized. With $k$ active blocks, model prior probability is $q^k(1-q)^{4-k}$. Training-only grouped CV Brier selects $s\in\{0.15,0.35,0.7,1,2.5\}$ and $q\in\{0.2,0.5,0.8\}$. This is empirical prior selection, not full integration over hyperparameters; $q=0.5$ recovers equal model priors.

At the maximum a posteriori estimate $\hat\beta_m$, let $H_m$ be the Hessian of the negative log posterior. Laplace evidence is
$$\log Z_m\approx\ell_m(\hat\beta_m)+\log p(\hat\beta_m\mid M_m)+\frac{k_m}{2}\log(2\pi)-\frac12\log|H_m|.$$
The normalized Gaussian prior is retained. Weights are computed stably as
$$w_m=\mathrm{softmax}_m(\log Z_m+\log p(M_m)).$$
The average is $p(y_*=1\mid x_*,D)\approx\sum_m w_m E_{N(\hat\beta_m,H_m^{-1})}[\sigma(x_{*m}^T\beta_m)]$; 20-point Gauss-Hermite quadrature integrates the Gaussian linear predictor.

This is **approximate Bayesian model averaging**, not averaging classifier accuracies, stacking, or an exact evidence calculation. Evidence weights are conditional on the candidate family, preprocessing and selected prior. They are not causal feature effects or guarantees of predictive skill.

## 5. Fit candidates, tune alternatives and freeze decisions
Compare BMA with Logistic Regression (`C`: .001, .01, .1, 1, 10, 100), Decision Tree (depth 1/2/4/6, leaf size 5/15/30, pruning alpha 0/.01), Random Forest (300 trees; depth 3/unrestricted, leaf size 5/15/30, feature fraction .5/1), and a training-prevalence baseline. All use the exact same grouped inner folds and the same held-out rows.

Select hyperparameters by mean inner-CV Brier (lower is better). Fit selected configurations on training only. Each model selects its F1 threshold on validation over .05 to .95 in .01 steps; ties favor the threshold closest to .5. Validation F1 is therefore optimistic. Validation Brier is not the hyperparameter selection score. Report test F1 at both the frozen threshold and .5.

No post-hoc calibration transformation is fitted: the required calibration check assesses the original probabilities. The same test set was previously published; no independent external validation is claimed, and test performance is never used to select a configuration.

In [ ]:
comparison = run_experiment()
display(comparison)
metadata = json.loads((ROOT / 'artifacts/metadata.json').read_text())
display(metadata['best_parameters'])

## 6. Shared-test comparison
ROC-AUC assesses ranking. F1 depends on threshold and prevalence. Brier is mean squared probability error; it combines calibration and discrimination, so a low Brier score alone does not establish perfect calibration. Log loss is also provided.

Training time includes CV search plus the selected refit; refit time is shown separately. Inference time is the mean of 30 warmed test-batch predictions divided by row count, including pipeline transformations but excluding loading/I/O. It is not single-request latency. No model is assumed to outperform the baseline.

In [ ]:
columns = ['algorithm', 'cv_brier', 'validation_brier', 'test_roc_auc', 'test_f1', 'test_f1_at_0_5',
           'test_brier', 'test_precision', 'test_recall', 'test_balanced_accuracy', 'threshold', 'training_seconds_including_search',
           'selected_fit_seconds', 'inference_ms_per_row_batch', 'interpretability']
display(comparison[columns].round(4))
display(Image(filename=str(ROOT / 'artifacts/comparison.png')))

## 7. Model weights and block inclusion
Candidate weights sum to one. Block inclusion is the sum of posterior model weights for models containing that block. Inspect whether the intercept-only model has material support: complex weather relationships may not be supported by these data.

In [ ]:
weights = pd.read_csv(ROOT / 'artifacts/bma_weights.csv')
assert np.isclose(weights.weight.sum(), 1)
assert (weights.gradient_max < 1e-4).all()
display(weights)
display(pd.read_csv(ROOT / 'artifacts/block_inclusion.csv'))
display(Image(filename=str(ROOT / 'artifacts/model_weights.png')))

## 8. Calibration and errors
Plot reliability in five fixed equal-width probability bins against the ideal diagonal. Persist counts and observed proportions for every occupied bin. Empty bins are omitted, not treated as zero. Small samples create noisy curves; this is a diagnostic, not evidence that all probabilities are reliable. ROC and calibration use the same held-out spatial rows for every model. No test-fitted calibrator feeds predictions back into evaluation.

In [ ]:
display(Image(filename=str(ROOT / 'artifacts/roc_calibration.png')))
display(pd.read_csv(ROOT / 'artifacts/calibration_bins.csv'))
display(Image(filename=str(ROOT / 'artifacts/bma_confusion.png')))
baseline = comparison.set_index('algorithm').loc['Prevalence Baseline', 'test_brier']
for _, row in comparison.iterrows():
    print(f"{row.algorithm}: test Brier {row.test_brier:.4f}; improvement over baseline {baseline - row.test_brier:+.4f}")

## 9. Load the application model and score a scenario
The app uses the same serialized pipelines, selected thresholds and data schema. Inputs are validated and extrapolation beyond training ranges is flagged. Uploaded batch CSVs are scored locally; downloadable results include each model's probability. Between-model disagreement is explicitly distinguished from a confidence interval.

A separate 90% latent-probability credible interval samples model indices and coefficient posteriors using 2,000 draws (seed 42). It is conditional on the Laplace approximation and candidate family; it does not cover dataset bias. The test metric 95% percentile intervals resample spatial cells 500 times (seed 2026) without refitting, so they describe conditional sampling variability only.

In [ ]:
import joblib
bundle = joblib.load(ROOT / 'artifacts/models.joblib')  # trusted, locally generated artifact only
scenario = pd.DataFrame([dict(temp=20.0, RH=45.0, wind=4.0, rain=0.0,
                              FFMC=90.0, DMC=110.0, DC=550.0, ISI=8.0, month='aug')])
scenario_X = feature_frame(scenario)
display(pd.DataFrame([{'algorithm': name, 'positive_recorded_area_probability': model.predict_proba(scenario_X)[0, 1],
                       'threshold': bundle['thresholds'][name]} for name, model in bundle['models'].items()]))
print('Launch from project folder: python -m streamlit run app.py')
bma_pipe = bundle['models']['BMA']
interval = bma_pipe.named_steps['model'].probability_interval(bma_pipe.named_steps['scale'].transform(scenario_X))[0]
print('BMA 90% latent-probability interval:', interval)
display(pd.read_csv(ROOT / 'artifacts/test_metric_intervals.csv'))

## 10. Interpretation, limitations and next steps
* This dataset supports positive recorded-area classification among fire records; it cannot validate daily ignition risk. Obtain representative fire and non-fire exposure records before making an occurrence claim.
* Only one small park dataset is available. Grouped spatial holdout reduces location leakage, but shared weather/event dependence across cells may remain. There are no full timestamps for chronological validation.
* All estimates are conditional on one prespecified split. Small test size and sparse probability bins limit precision. Spatial bootstrap metric intervals are reported conditional on fitted models; no fresh external validation is claimed. The original test set was previously published.
* BMA uses a restricted logistic candidate family and Laplace approximations; five prior scales and three block-inclusion priors are assessed within training CV. A wider family or exact posterior inference could change the weights.
* F1 thresholds optimize a statistical criterion on a small validation set, not firefighting costs or policy. The interface is an educational research demonstration.
* If ranking or Brier is weak relative to baseline, report it as measured. Do not change the held-out set or manufacture higher accuracy.

### Files produced
`artifacts/` contains the common split assignments, CV search results, comparison table, test probabilities, calibration-bin counts, model weights, block inclusion, plots, model bundle and environment/data-hash metadata. `python -m wildfire.deliverables` also generates the required short PDF report and 12-slide editable PPTX. `app.py` provides the working interface; `tests/` checks spatial separation, leakage, BMA normalization, inference stability and application prediction.

### References
* [UCI Forest Fires](https://archive.ics.uci.edu/dataset/162/forest%2Bfires)
* [scikit-learn probability calibration documentation](https://scikit-learn.org/stable/modules/calibration.html)
* [SciPy minimize documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)
